In [6]:
# Step 1. Setup
import pandas as pd
import os

base_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_exports"
map_path  = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_mapping"
output_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"

os.makedirs(output_path, exist_ok=True)

# Mapping files
rxnorm_map = pd.read_csv(f"{map_path}/inputevents_to_rxnorm.csv")
output_map = pd.read_csv(f"{map_path}/outputevents_to_loinc.csv")

# ICU stay file (for stay_id → hadm_id mapping)
icustays = pd.read_csv(f"{base_path}/icu_icustays.csv")[["stay_id", "hadm_id"]]

outfile = f"{output_path}/IN_OUT.csv"

print("✅ Setup complete")


✅ Setup complete


In [7]:
# Step 2. Process Inputevents → RxNorm
inputevents_path = f"{base_path}/icu_inputevents.csv"

inputs_final_list = []

chunk_iter = pd.read_csv(
    inputevents_path,
    usecols=["stay_id", "starttime", "itemid", "ordercategoryname"],
    chunksize=500000
)

for i, chunk in enumerate(chunk_iter):
    merged = chunk.merge(
        rxnorm_map,
        left_on="itemid",
        right_on="itemid (omop_source_code)",
        how="left"
    )
    merged["service_ts"] = merged["starttime"]
    merged["order_ts"] = merged["starttime"]

    merged = merged.rename(columns={
        "omop_concept_name": "order_clinical_desc",
        "ordercategoryname": "order_catalog_desc"
    })[["stay_id", "service_ts", "order_ts", "order_clinical_desc", "order_catalog_desc"]]

    inputs_final_list.append(merged)
    print(f"Processed inputevents chunk {i+1}, shape={merged.shape}")

inputs_final = pd.concat(inputs_final_list, ignore_index=True)

# join hadm_id
inputs_final = inputs_final.merge(icustays, on="stay_id", how="left")
inputs_final = inputs_final.rename(columns={"hadm_id": "csn"})
inputs_final = inputs_final.drop(columns=["stay_id"])

print("✅ Inputs processed:", inputs_final.shape)


Processed inputevents chunk 1, shape=(851739, 5)
Processed inputevents chunk 2, shape=(857498, 5)
Processed inputevents chunk 3, shape=(848977, 5)
Processed inputevents chunk 4, shape=(854827, 5)
Processed inputevents chunk 5, shape=(855578, 5)
Processed inputevents chunk 6, shape=(854790, 5)
Processed inputevents chunk 7, shape=(853914, 5)
Processed inputevents chunk 8, shape=(854336, 5)
Processed inputevents chunk 9, shape=(854419, 5)
Processed inputevents chunk 10, shape=(858022, 5)
Processed inputevents chunk 11, shape=(857414, 5)
Processed inputevents chunk 12, shape=(848765, 5)
Processed inputevents chunk 13, shape=(851704, 5)
Processed inputevents chunk 14, shape=(853493, 5)
Processed inputevents chunk 15, shape=(856648, 5)
Processed inputevents chunk 16, shape=(858832, 5)
Processed inputevents chunk 17, shape=(854419, 5)
Processed inputevents chunk 18, shape=(854613, 5)
Processed inputevents chunk 19, shape=(856621, 5)
Processed inputevents chunk 20, shape=(853889, 5)
Processed

In [8]:
# Step 3. Process Outputevents → LOINC
outputevents_path = f"{base_path}/icu_outputevents.csv"

outputs_final_list = []

chunk_iter = pd.read_csv(
    outputevents_path,
    usecols=["stay_id", "charttime", "itemid"],
    chunksize=500000
)

for i, chunk in enumerate(chunk_iter):
    merged = chunk.merge(
        output_map,
        left_on="itemid",
        right_on="itemid (omop_source_code)",
        how="left"
    )
    
    merged["service_ts"] = merged["charttime"]
    merged["order_ts"] = merged["charttime"]

    merged = merged.rename(columns={
        "omop_concept_name": "order_clinical_desc",
        "category": "order_catalog_desc"
    })[["stay_id", "service_ts", "order_ts", "order_clinical_desc", "order_catalog_desc"]]

    outputs_final_list.append(merged)
    print(f"Processed outputevents chunk {i+1}, shape={merged.shape}")

outputs_final = pd.concat(outputs_final_list, ignore_index=True)

# join hadm_id
outputs_final = outputs_final.merge(icustays, on="stay_id", how="left")
outputs_final = outputs_final.rename(columns={"hadm_id": "csn"})
outputs_final = outputs_final.drop(columns=["stay_id"])

print("✅ Outputs processed:", outputs_final.shape)


Processed outputevents chunk 1, shape=(500000, 5)
Processed outputevents chunk 2, shape=(500000, 5)
Processed outputevents chunk 3, shape=(500000, 5)
Processed outputevents chunk 4, shape=(500000, 5)
Processed outputevents chunk 5, shape=(500000, 5)
Processed outputevents chunk 6, shape=(500000, 5)
Processed outputevents chunk 7, shape=(500000, 5)
Processed outputevents chunk 8, shape=(500000, 5)
Processed outputevents chunk 9, shape=(500000, 5)
Processed outputevents chunk 10, shape=(500000, 5)
Processed outputevents chunk 11, shape=(359395, 5)
✅ Outputs processed: (5359395, 5)


In [9]:
# Step 4. Combine Inputs + Outputs
in_out_final = pd.concat(
    [inputs_final, outputs_final],
    ignore_index=True
)

cols = ["csn", "service_ts", "order_ts", "order_clinical_desc", "order_catalog_desc"]
in_out_final = in_out_final[cols]

print("✅ Combined shape:", in_out_final.shape)
in_out_final.head()


✅ Combined shape: (24074888, 5)


,csn,service_ts,order_ts,order_clinical_desc,order_catalog_desc
0,25062453,2141-04-28 16:08:00,2141-04-28 16:08:00,cefepime Injection,08-Antibiotics (IV)
1,25062453,2141-04-28 16:08:00,2141-04-28 16:08:00,cefepime Injection,08-Antibiotics (IV)
2,25062453,2141-04-28 16:08:00,2141-04-28 16:08:00,cefepime Injection,08-Antibiotics (IV)
3,25062453,2141-04-28 16:08:00,2141-04-28 16:08:00,cefepime Injection,08-Antibiotics (IV)
4,25062453,2141-04-28 16:50:00,2141-04-28 16:50:00,NaN,01-Drips


In [10]:
# Step 5. Save final file
in_out_final.to_csv(outfile, index=False)
print(f"✅ Saved IN_OUT file to {outfile}")

✅ Saved IN_OUT file to /hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/IN_OUT.csv
